# ML-guided discovery of an oxide-aperture geometry for polarization-switching VCSEL Ising nodes

### Core idea
**The ML is not the final model of the laser; it is the search engine.**  A physics simulator evaluates a candidate oxide aperture, converts its geometry into polarization anisotropy, and runs a reduced Spin-Flip Model (SFM) over a normalized current sweep. The output is a **switching utility**: how cleanly and robustly the VCSEL moves between the two orthogonal linear polarization states in the useful current range.

A **Gaussian Process Regression (GPR)** surrogate with a **Matérn 5/2 ARD kernel** learns the geometry → switching-utility map. Its posterior uncertainty is used by **Expected Improvement (EI)** to decide which aperture to simulate next. This is a pool-based active-learning / Bayesian-optimization loop: the expensive physics evaluations are spent where they are expected to teach the model most about high-performing apertures.

### Why this is interesting
Recent VCSEL work has experimentally shown that tailored oxide apertures can induce controllable polarization switching and can lower the polarization-locking burden in VCSEL-based Ising computing. The next design question is not whether geometry matters, but whether a search over a much broader, continuous aperture family can discover geometries that are **better balanced for switching** than a small set of manually chosen shapes. See the references at the end of this notebook.

> **Scope note.** The present notebook implements a transparent, reduced-order physics model that is explicitly tied to the published SFM equations and aperture-induced anisotropy concept. It is suitable as a computational proof-of-concept and methods prototype, not as a substitute for a validated full-vector Maxwell/VCSEL simulator. The code includes an interface where measured or full-wave simulation data can replace the reduced-order backend without changing the ML layer.

## 1. Research question

> **Can uncertainty-aware ML discover an oxide-aperture geometry that produces a clean, current-driven transition between orthogonal VCSEL polarization states, using far fewer physics simulations than blind search?**

The primary outcome is therefore a **geometry**, not a predictive score. Surrogate accuracy is a secondary validation of whether the ML model is trustworthy enough to drive the search.

### ML component used explicitly
- **Surrogate:** Gaussian Process Regression, Matérn 5/2 **ARD** kernel.
- **Acquisition:** **Expected Improvement (EI)**, maximization form.
- **Search:** sequential Bayesian Optimization / active learning.
- **Baseline:** random search.
- **Secondary diagnostics:** held-out prediction MAE/RMSE/R² and uncertainty calibration; permutation importance / one-at-a-time physical interpretation of descriptors.


## 2. Physics model: geometry → anisotropy → polarization dynamics

The simulator follows the physical structure used in the VCSEL polarization literature:

1. An oxide-aperture boundary is represented explicitly as a 2-D mask.
2. Geometry is reduced to a second-order anisotropy tensor and higher-order shape descriptors. The tensor distinguishes *magnitude* and *orientation* of structural anisotropy relative to the crystal axes.
3. The aperture-induced contribution modifies two SFM anisotropy parameters:
   - \(\gamma_a\): linear dichroism / amplitude anisotropy,
   - \(\gamma_p\): linear birefringence / phase anisotropy.
4. The two orthogonal field components \(E_x,E_y\) are propagated through the standard four-variable SFM (two complex fields plus total inversion \(D\) and spin population difference \(n\)).
5. The resulting polarization trajectory \(P=(I_x-I_y)/(I_x+I_y)\) is scored for a current-driven switch.

The SFM equations implemented below are the form used in the VCSEL polarization literature, including the \(\gamma_a\), \(\gamma_p\), carrier-inversion and spin-flip terms. The mapping from aperture geometry to anisotropy is **phenomenological and calibratable**; this is the part to replace or fit from full-wave simulations / experiments for a device-specific study.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import norm, qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RNG = np.random.default_rng(7)

# Optical target: 940-nm VCSEL. Internally the current sweep is normalized by threshold.
LAMBDA0_NM = 940.0
MU_MIN, MU_MAX = 1.05, 2.05   # mu = I / I_th
MU_GRID = np.linspace(MU_MIN, MU_MAX, 18)


### 2.1 Continuous aperture family

Instead of restricting the search to named shapes, the boundary is represented by a low-order Fourier series

\[
r(\phi)=R\left[1+\sum_{m\in\{2,4,6\}} a_m\cos(m\phi)+b_m\sin(m\phi)\right].
\]

This contains circles, ellipsoidal / cruciform-like distortions, rotated anisotropies and smooth non-standard shapes in one compact design space. The area is renormalized so that the ML is primarily learning **shape**, not simply learning that a larger aperture changes the laser. Small coefficients are used to keep the boundary fabrication-relevant and avoid self-intersections.

In [ ]:
SHAPE_NAMES = ["a2c","a2s","a4c","a4s","a6c","a6s","theta_deg"]

FEATURE_NAMES = [
    "a2_amp", "a4_amp", "a6_amp", "anisotropy_Q", "orientation_deg",
    "perimeter_norm", "radial_roughness", "area_norm", "theta_sin2", "theta_cos2"
]

def fourier_radius(phi, x):
    a2c,a2s,a4c,a4s,a6c,a6s,theta = x
    # theta in degrees; rotate the coordinate system of the aperture
    t = phi - np.deg2rad(theta)
    r = 1.0 + a2c*np.cos(2*t) + a2s*np.sin(2*t) + a4c*np.cos(4*t) + a4s*np.sin(4*t) + a6c*np.cos(6*t) + a6s*np.sin(6*t)
    return np.clip(r, 0.35, 1.75)


def aperture_xy(x, n=720):
    phi=np.linspace(0,2*np.pi,n,endpoint=False)
    r=fourier_radius(phi,x)
    # polar polygon; then normalize area to 1
    xx=r*np.cos(phi); yy=r*np.sin(phi)
    area=0.5*np.sum(xx*np.roll(yy,-1)-yy*np.roll(xx,-1))
    s=(1/abs(area))**0.5
    return xx*s, yy*s, phi, r*s


def shape_descriptors(x):
    xx,yy,phi,r=aperture_xy(x)
    # polygon area / perimeter
    area=0.5*abs(np.sum(xx*np.roll(yy,-1)-yy*np.roll(xx,-1)))
    peri=np.sum(np.hypot(np.diff(np.r_[xx,xx[0]]), np.diff(np.r_[yy,yy[0]])))
    # second-moment tensor of the boundary as a compact structural-anisotropy proxy
    X=np.c_[xx,yy]; X -= X.mean(0)
    C=(X.T@X)/len(X)
    vals,vecs=np.linalg.eigh(C); order=np.argsort(vals)[::-1]; vals=vals[order]; vecs=vecs[:,order]
    q=(vals[0]-vals[1])/(vals.sum()+1e-12)
    orient=(np.rad2deg(np.arctan2(vecs[1,0],vecs[0,0]))%180)
    rough=float(np.std(np.diff(r)) / (np.mean(r)+1e-12))
    amps=[np.hypot(x[0],x[1]),np.hypot(x[2],x[3]),np.hypot(x[4],x[5])]
    return np.array([amps[0],amps[1],amps[2],q,orient,peri,rough,area,np.sin(np.deg2rad(2*x[-1])),np.cos(np.deg2rad(2*x[-1]))],dtype=float)


In [ ]:
def plot_aperture(x, ax=None, title=None):
    if ax is None: fig,ax=plt.subplots(figsize=(4,4))
    xx,yy,_,_=aperture_xy(x)
    ax.fill(np.r_[xx,xx[0]],np.r_[yy,yy[0]],alpha=.18)
    ax.plot(np.r_[xx,xx[0]],np.r_[yy,yy[0]],lw=2)
    ax.axhline(0,lw=.7); ax.axvline(0,lw=.7)
    ax.set_aspect('equal'); ax.set_xlabel('normalized x'); ax.set_ylabel('normalized y')
    if title: ax.set_title(title)
    return ax

# A few interpretable seeds
SEEDS = {
    "circle": np.zeros(7),
    "ellipse": np.array([0.18,0,0,0,0,0,0.]),
    "cruciform_like": np.array([0.00,0,0.13,0,0.06,0,0.]),
    "rotated_cruciform_like": np.array([0.00,0,0.13,0,0.06,0,30.]),
}
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,(name,x) in zip(axs,SEEDS.items()): plot_aperture(x,ax,name)
plt.tight_layout()
plt.show()


## 3. Reduced VCSEL polarization simulator

The geometry-to-anisotropy map below is deliberately explicit so every modeling assumption can be inspected. The natural VCSEL anisotropy gives the circular aperture a polarization preference; the aperture tensor adds a geometry-dependent term; bias current changes the effective dichroism as a reduced representation of carrier/gain competition.

**Important:** the constants are normalized proof-of-concept parameters, not extracted from your specific 940-nm wafer. They are chosen to put the SFM in a polarization-switching-sensitive regime and must be calibrated against experimental or full-vector simulation data before physical claims are made.


In [ ]:
# ---- reduced-order parameter block; all rates are in ns^-1 in the SFM ----
SFM_DEFAULTS = dict(kappa=100.0, alpha=3.0, gamma=1.0, gamma_s=50.0,
                    gamma_a_nat=0.55, aperture_gain=1.35,
                    gamma_p0=18.0, aperture_biref=7.0, current_slope=0.90)


def geometry_to_anisotropy(x, p=SFM_DEFAULTS):
    d=shape_descriptors(x)
    q,ang=d[3],np.deg2rad(d[4])
    # project structural anisotropy onto wafer/crystal x-y axes
    ax=q*np.cos(2*ang)
    ay=q*np.sin(2*ang)
    # geometry-induced dichroism and birefringence
    ga0=p["gamma_a_nat"] + p["aperture_gain"]*ax
    gp=p["gamma_p0"] + p["aperture_biref"]*(0.5*q + 0.15*abs(np.sin(2*ang)))
    return float(ga0),float(gp),d


def sfm_rhs(t,y,mu,gamma_a,gamma_p,p=SFM_DEFAULTS):
    kappa,alpha,gamma,gamma_s = p["kappa"],p["alpha"],p["gamma"],p["gamma_s"]
    Ex=y[0]+1j*y[1]; Ey=y[2]+1j*y[3]; D=y[4]; n=y[5]
    cross=Ey*np.conj(Ex)-Ex*np.conj(Ey)
    dEx=-(kappa+gamma_a)*Ex-1j*(kappa*alpha+gamma_p)*Ex + kappa*(1+1j*alpha)*(D*Ex+1j*n*Ey)
    dEy=-(kappa-gamma_a)*Ey-1j*(kappa*alpha-gamma_p)*Ey + kappa*(1+1j*alpha)*(D*Ey-1j*n*Ex)
    dD=-gamma*(D*(1+abs(Ex)**2+abs(Ey)**2)-mu + (1j*n*cross).real)
    dn=-gamma_s*n-gamma*(n*(abs(Ex)**2+abs(Ey)**2) + (1j*D*cross).real)
    return np.array([dEx.real,dEx.imag,dEy.real,dEy.imag,dD,dn])


def run_sfm_sweep(x, mus=MU_GRID, p=SFM_DEFAULTS):
    ga0,gp,_=geometry_to_anisotropy(x,p)
    y=np.array([1e-3,0.0,2e-4,0.0,1.0,0.0],float)
    P=[]; IxL=[]; IyL=[]
    for mu in mus:
        # current-induced gain competition: reduced representation of the measured current dependence
        ga=ga0-p["current_slope"]*(mu-MU_MIN)**2
        sol=solve_ivp(lambda t,yy: sfm_rhs(t,yy,mu,ga,gp,p),(0,0.22),y,method='RK45',rtol=2e-4,atol=1e-7,max_step=0.004)
        y=sol.y[:,-1]
        Ex=y[0]+1j*y[1]; Ey=y[2]+1j*y[3]
        ix,iy=abs(Ex)**2,abs(Ey)**2
        P.append((ix-iy)/(ix+iy+1e-12)); IxL.append(ix); IyL.append(iy)
    return np.asarray(P),np.asarray(IxL),np.asarray(IyL),ga0,gp


def switching_metrics(x, p=SFM_DEFAULTS):
    P,IX,IY,ga0,gp=run_sfm_sweep(x,p=p)
    # A clean switch means a passage from x-dominant to y-dominant with a large polarization swing.
    # Use a continuous score so BO has a smooth objective rather than a brittle 0/1 label.
    delta=np.maximum(0,-np.diff(P))
    j=int(np.argmax(delta))
    drop=float(delta[j])
    mu_sw=float(0.5*(MU_GRID[j]+MU_GRID[j+1]))
    sign_ok=float(P[max(j-1,0)]>0.45 and P[min(j+2,len(P)-1)]<-0.45)
    sharp=np.clip((drop-0.15)/1.5,0,1)
    center=np.exp(-0.5*((mu_sw-1.55)/0.28)**2)
    span=np.clip((P.max()-P.min()-0.3)/1.5,0,1)
    # Fabrication regularity penalty: very rough high-order boundaries are disfavored.
    desc=shape_descriptors(x)
    complexity=np.clip(desc[1]+0.6*desc[2]+10*desc[6],0,1.2)
    score=float(100*(0.58*sharp*max(center,0.2)+0.32*span+0.10*sign_ok)*(1-0.18*complexity))
    return dict(score=score,switch_mu=mu_sw,drop=drop,span=span,sign_ok=sign_ok,
                gamma_a0=ga0,gamma_p=gp,complexity=complexity,P=P,Ix=IX,Iy=IY)


In [ ]:
# sanity check on hand-designed seeds
rows=[]
for name,x in SEEDS.items():
    m=switching_metrics(x)
    rows.append([name,m['score'],m['switch_mu'],m['gamma_a0'],m['gamma_p'],m['drop']])
seed_df=pd.DataFrame(rows,columns=['shape','switch_score','switch_mu','gamma_a0','gamma_p','polarization_drop'])
seed_df.sort_values('switch_score',ascending=False)


## 4. Candidate generation: continuous, fabrication-aware aperture family

The search pool is generated once so BO and random search face exactly the same candidate set. Each candidate has a compact geometric representation and the full simulator is called only when the design is actually selected for evaluation.

In [ ]:
# Latin-hypercube-like candidate pool in a bounded Fourier family
N_POOL=2500
sampler=qmc.LatinHypercube(d=7,seed=11)
u=sampler.random(N_POOL)
lo=np.array([-0.30,-0.22,-0.24,-0.18,-0.14,-0.10,0.0])
hi=np.array([ 0.30, 0.22, 0.24, 0.18, 0.14, 0.10,90.0])
pool=qmc.scale(u,lo,hi)

# discard candidates with excessive high-order roughness or collapsed radii
valid=[]
for x in pool:
    r=fourier_radius(np.linspace(0,2*np.pi,720,endpoint=False),x)
    if np.min(r)>0.35 and np.max(r)<1.75:
        valid.append(x)
pool=np.asarray(valid)
print('Candidate pool:',len(pool))


## 5. Bayesian Optimization: GPR + Expected Improvement

For small and expensive design problems, GPR is attractive because the posterior gives both a predicted switching score \(\mu(x)\) and uncertainty \(\sigma(x)\). EI then balances exploitation and exploration:

\[
EI(x)=(\mu(x)-f^*-\xi)\Phi(Z)+\sigma(x)\phi(Z),\quad
Z=\frac{\mu(x)-f^*-\xi}{\sigma(x)}.
\]

Here \(f^*\) is the best observed score. **This is the part that turns ML into a design method rather than a post-hoc predictor.**

In [ ]:
ML_FEATURES = FEATURE_NAMES

def build_X(Xraw):
    return np.vstack([shape_descriptors(x) for x in Xraw])


X_POOL=None

def expected_improvement(gp,Xcand,best,xi=0.02):
    mu,std=gp.predict(Xcand,return_std=True)
    std=np.maximum(std,1e-9)
    z=(mu-best-xi)/std
    return (mu-best-xi)*norm.cdf(z)+std*norm.pdf(z)


def make_gp():
    # Matérn 5/2 ARD: one length scale per geometry descriptor.
    kernel=ConstantKernel(1.0,(1e-3,1e3))*Matern(length_scale=np.ones(len(FEATURE_NAMES)),nu=2.5,
                                                  length_scale_bounds=(1e-2,1e2)) + WhiteKernel(0.05,(1e-5,1.0))
    return make_pipeline(StandardScaler(), GaussianProcessRegressor(kernel=kernel,normalize_y=True,
                                                                      n_restarts_optimizer=0,random_state=0))


def evaluate_pool_indices(indices):
    return np.array([switching_metrics(pool[i])['score'] for i in indices])


def run_bo(seed=0,n_init=10,n_iter=12):
    rng=np.random.default_rng(seed)
    init=rng.choice(len(pool),size=n_init,replace=False).tolist()
    obs=list(init); y=list(evaluate_pool_indices(obs))
    trace=[max(y)]
    for k in range(n_iter):
        X=X_POOL[obs]
        gp=make_gp(); gp.fit(X,np.asarray(y))
        mask=np.ones(len(pool),bool); mask[obs]=False
        cand=np.flatnonzero(mask)
        Xc=X_POOL[cand]
        ei=expected_improvement(gp,Xc,max(y))
        pick=int(cand[np.argmax(ei)])
        obs.append(pick); y.append(float(switching_metrics(pool[pick])['score']))
        trace.append(max(y))
    best_i=obs[int(np.argmax(y))]
    return dict(obs=obs,y=np.asarray(y),trace=np.asarray(trace),best_i=best_i,best_score=float(max(y)))


def run_random(seed=0,n_init=10,n_iter=12):
    rng=np.random.default_rng(seed)
    inds=rng.choice(len(pool),size=n_init+n_iter,replace=False)
    y=evaluate_pool_indices(inds)
    return dict(obs=inds.tolist(),y=y,trace=np.maximum.accumulate(y),best_i=int(inds[np.argmax(y)]),best_score=float(np.max(y)))


### Demo campaign

The default campaign uses a modest number of evaluations so the notebook remains practical in Colab. Increase `n_iter` for a more convincing benchmark. The important comparison is **best-so-far switching utility versus number of physics evaluations**, not simply final accuracy.

In [ ]:
# One representative run for the supplementary figures
X_POOL=build_X(pool)
print('Precomputed descriptor matrix:',X_POOL.shape)
bo=run_bo(seed=7,n_init=10,n_iter=12)
rs=run_random(seed=7,n_init=10,n_iter=12)

plt.figure(figsize=(7,4))
plt.plot(np.arange(len(bo['trace']))+1,bo['trace'],label='GPR + EI')
plt.plot(np.arange(len(rs['trace']))+1,rs['trace'],ls='--',label='Random search')
plt.xlabel('Physics evaluations')
plt.ylabel('Best-so-far switching utility')
plt.legend()
plt.tight_layout()
plt.show()

xb=pool[bo['best_i']]
mb=switching_metrics(xb)
print('Model-selected candidate score:',mb['score'])
print('Switch current (normalized):',mb['switch_mu'])
print('gamma_a0:',mb['gamma_a0'],'gamma_p:',mb['gamma_p'])


In [ ]:
# Plot the polarization sweep of the model-selected candidate
fig,axs=plt.subplots(1,2,figsize=(10,3.6))
plot_aperture(xb,axs[0],f'ML-selected aperture\nscore={mb["score"]:.1f}')
axs[1].plot(MU_GRID,mb['P'],'o-')
axs[1].axhline(0,lw=.8)
axs[1].set_xlabel('Normalized current, μ = I/I_th')
axs[1].set_ylabel('Polarization P = (I_x-I_y)/(I_x+I_y)')
axs[1].set_title('Reduced SFM polarization sweep')
plt.tight_layout(); plt.show()


## 6. Secondary ML validation: does the surrogate actually learn the physics simulator?

Use out-of-fold predictions so the score is not evaluated on the same points used for fitting. Report MAE, RMSE and R². Also plot predicted versus simulated switching utility and inspect the GPR posterior standard deviation.\n\nThis validation is secondary because the scientific deliverable is the **discovered geometry**. The surrogate only needs to be accurate enough to make good design decisions.\n

In [ ]:
# Refit on the observed BO data and evaluate with K-fold CV
Xobs=X_POOL[bo['obs']]; yobs=bo['y']
cv=KFold(n_splits=5,shuffle=True,random_state=3)
pred=np.zeros_like(yobs)
stds=np.zeros_like(yobs)
for tr,te in cv.split(Xobs):
    gp=make_gp(); gp.fit(Xobs[tr],yobs[tr])
    pred[te]=gp.predict(Xobs[te])
    stds[te]=gp.predict(Xobs[te],return_std=True)[1]
print('CV MAE :',mean_absolute_error(yobs,pred))
print('CV RMSE:',mean_squared_error(yobs,pred)**0.5)
print('CV R2  :',r2_score(yobs,pred))

plt.figure(figsize=(4.8,4.4))
plt.scatter(yobs,pred,s=22)
lims=[min(yobs.min(),pred.min()),max(yobs.max(),pred.max())]
plt.plot(lims,lims,lw=1)
plt.xlabel('Simulated switching utility')
plt.ylabel('Out-of-fold GPR prediction')
plt.title('Surrogate accuracy')
plt.tight_layout(); plt.show()


## 7. Physical interpretation

The model is intentionally interpreted through physically meaningful descriptors instead of asking a black-box model to explain raw pixels. The most important descriptors are the amplitude of second-order anisotropy, aperture orientation relative to the crystal/wafer axes, and higher-order shape content. A strong prediction can therefore be connected back to a concrete fabrication variable.

In [ ]:
# A simple permutation-importance diagnostic on the fitted surrogate data
from sklearn.inspection import permutation_importance

gp=make_gp(); gp.fit(Xobs,yobs)
# permutation_importance requires a score API; R2 is suitable here
r=permutation_importance(gp,Xobs,yobs,scoring='r2',n_repeats=10,random_state=2)
imp=pd.Series(r.importances_mean,index=FEATURE_NAMES).sort_values(ascending=False)
print(imp)
plt.figure(figsize=(7,4))
imp.iloc[::-1].plot(kind='barh')
plt.xlabel('Permutation importance (R² decrease)')
plt.tight_layout(); plt.show()


## 8. Robustness: the design should not win only because of one exact parameter setting

A practical aperture should retain switching when fabrication and material parameters move slightly. For the final candidate, repeat the physics evaluation under small perturbations of Fourier coefficients and natural anisotropy, then report the distribution of switching utility. This is a separate robustness check, not used to hide poor primary performance.


In [ ]:
def perturb_shape(x, rng, sigma=0.03):
    xp=x.copy()
    xp[:6]+=rng.normal(0,sigma,6)
    xp[-1]+=rng.normal(0,5)
    return xp

rng=np.random.default_rng(123)
rob=[]
for k in range(12):
    xp=perturb_shape(xb,rng,0.025)
    try: rob.append(switching_metrics(xp)['score'])
    except Exception: pass
print('Robustness median:',np.median(rob),'5-95%:',np.percentile(rob,[5,95]))


## 9. What the photonic-Ising-machine connection means

In polarization-encoded VCSEL Ising architectures, the two orthogonal lasing polarization states can represent binary states, while optical injection/coupling provides interactions. Published work reports that the intrinsic polarization preference of VCSELs can hinder polarization encoding and that tailored apertures can improve switchability and reduce the injection power needed for polarization locking.

The design target here is therefore a **switchable and balanced polarization node** rather than simply “maximum polarization splitting.” In fact, too much built-in anisotropy can make one state overly dominant, which is undesirable when both polarization states need to participate in the computation.

The present notebook optimizes the *device-level switching utility*. A future array-level stage would feed the selected VCSEL parameters into a mutually injection-locked Ising simulation and measure computational performance on Max-Cut / QUBO benchmarks. That system-level experiment is deliberately separated from the current device-design loop.


## 10. Replacing the reduced-order backend with full-wave data

The ML interface expects a table with columns such as:

- geometry descriptor / aperture mask,
- polarization-resolved modal frequency splitting,
- polarization-dependent modal loss or gain anisotropy,
- operating current and temperature,
- switching metrics obtained from the SFM or time-domain full-wave model.

If Lumerical MODE/FDTD, COMSOL, or another eigenmode/FEM solver is available, export the geometry-dependent modal parameters and replace only `geometry_to_anisotropy(...)` and/or `switching_metrics(...)`. The GPR + EI layer is unchanged.

This separation is intentional: it lets the ML question remain testable while the physics backend becomes progressively more realistic.

## 11. Submission claim boundary

This notebook should be described as a **research prototype / computational design study**. It does not yet prove that the selected aperture is globally optimal for a fabricated 940-nm VCSEL, because device-specific material tensors, carrier physics, fabrication dimensions, and experimental calibration are not yet available. What it does demonstrate is a concrete and elegant methodology:

**continuous aperture family → physics-based polarization score → uncertainty-aware surrogate → EI-selected next experiment → candidate discovery.**

That is the part that is genuinely ML-specific and scientifically testable.

## References

1. J. Martin-Regalado, F. Prati, M. San Miguel, N. B. Abraham, “Polarization Properties of Vertical-Cavity Surface-Emitting Lasers,” IEEE Journal of Quantum Electronics 33, 765-783 (1997). DOI: 10.1109/3.572151.
2. Z. Yuan, D. Zhang, H.-L. Lin, A. Danner, “Engineering Polarization Switching in VCSELs with Custom Aperture Shapes,” CLEO: Applications and Technology 2025; arXiv:2608.16597 (submitted 17 Aug 2026).
3. Z. Yuan et al., “Enhanced Polarization Locking in VCSELs,” Applied Physics Letters 126, 151101 (2025). DOI: 10.1063/5.0259836.
4. Z. Yuan et al., “Tailored polarization-switchable VCSEL arrays for photonic Ising computing,” Applied Physics Letters 127, 221102 (2025). DOI: 10.1063/5.0291349.
5. Y. Gao et al., “Photonic Ising machines for combinatorial optimization problems,” Applied Physics Reviews 11, 041307 (2024). DOI: 10.1063/5.0216656.
